# 01. External Baselines and Design Deltas

- Parent issue: `#16`
- Wave: `A`
- Snapshot date: `2026-04-20`
- Status: `evidence-complete, gated on #14`
- Upstream gate: `#14` constraints freeze (see `notebooks/00_competition_constraints_and_rules.ipynb`)
- Downstream consumers: `#19` baseline eval + normalization, `#20` submission packaging, `#23` solver, `#25` HPC SFT runbook
- Summary: extract concrete facts from the `konbu17` baseline (already vendored under `data/external/konbu17/cells/`), compare them to this repo's current architecture and plan assumptions, and emit an `Adopt / Reject / Gate` matrix. Any decision that depends on the Kaggle scoring contract or base model identity is explicitly **GATE (BLOCKED by #14)** — this notebook never *adopts* those values.

This notebook is **spec + probe + export only**. It does not train, does not run inference, and does not hit the network. All evidence is repo-local; external URLs are listed in the Sources section but are not fetched.


## Audience and Why It Matters

**Who reads this:**

- Notebook authors for Waves B–D (`#17–#25`) who need to know which konbu17 design choices are safe to borrow *now* and which are still blocked on Kaggle rules capture.
- Reviewers (human + architecture) who need a single place that cites the konbu17 source files line-by-line before any design change lands in `docs/architecture/ARCHITECTURE.md`.
- Future agents re-running Wave A: the notebook encodes the adoption rationale so nobody has to re-derive it from plan doc text.

**Why it matters:**

- The konbu17 baseline is the only end-to-end Nemotron reasoning pipeline the repo has real code for. Blindly porting its choices would (a) hardcode a 30B base model that may not match the Kaggle evaluator, (b) bake in a `\boxed{}`/`</think>` output contract that may not match Kaggle scoring, and (c) inherit Kaggle-runtime-specific environment hacks (Triton wheel, `ptxas-blackwell`, `mamba_ssm` / `causal_conv1d`) that will not run on generic hosts.
- Ignoring konbu17 is equally wrong: it already proves a provisional LoRA rank, a provisional 9-module target set, and a minimal `submission.zip` layout that our `#20` packaging plan relies on.
- This notebook's job is to make the `Adopt / Reject / Gate` split explicit, cite every row to a repo-local file, and make the `#14` gates visible so nobody can silently promote a `Gate` row to `Adopt`.

**Non-technical summary:** We are not copying a Kaggle winner's notebook. We are listing what they did, comparing it to what this project currently assumes, and marking the disagreements with the rule: if the decision depends on Kaggle scoring or the exact evaluator base model, we wait for `#14` to capture those rules before adopting anything.


## Decision / Hypothesis

**Hypothesis:** the konbu17 repo snapshot under `data/external/konbu17/cells/` contains enough code to recover a safe provisional baseline for (a) LoRA rank, (b) LoRA target modules, and (c) submission zip layout — but **not** enough to resolve the base model identity, the scoring contract, or the `\boxed{}` / `<think>` output format. Those three depend on Kaggle rules text that only `#14` can capture.

**Decision rule for every row in the delta matrix below:**

- If the konbu17 choice can be reproduced on any reasonable host and does not depend on Kaggle rules semantics → `Adopt (provisional)` with a konbu17 citation.
- If the konbu17 choice depends on the Kaggle evaluator's identity or scoring behavior → `Gate (BLOCKED by #14)`. Never `Adopt` until `#14` is resolved.
- If the konbu17 choice is Kaggle-runtime-specific (Blackwell `ptxas`, `mamba_ssm` wheel, priority-upweight CSV paths) and not portable → `Reject for portability` with a note that the *idea* may still be relevant on HPC.

**Non-goals:**

- No training run, no inference, no HuggingFace download, no `src/` or tests edits.
- No GitHub API calls.
- No recommendation for final hyperparameters; production hyperparameters must come after `#14` clears.


## Environment and Reproduction

- Python: 3.11+ (only uses the standard library plus optional `pandas` for rendering).
- Run from the repo root: `jupyter nbconvert --to notebook --execute notebooks/01_external_baselines_and_design_deltas.ipynb --output /tmp/out_01.ipynb`.
- No network calls. The notebook reads only repo-local files under `data/external/konbu17/cells/` and writes to `data/eval/`.
- Output file: `data/eval/baselines_delta_matrix.json` (machine-readable delta matrix + provenance).
- This notebook is a sibling of `notebooks/00_competition_constraints_and_rules.ipynb`. The `Gate` rows below reference the `BLOCKED` rows exported by that notebook to `data/eval/competition_constraints.json` — do not re-derive those decisions here.


In [ ]:
"""Setup cell: imports, repo-root resolution, konbu17 cell paths.

The notebook is designed to run cleanly from either the repo root or the
`notebooks/` directory under nbconvert. We resolve `REPO_ROOT` deterministically
and then derive every other path from it.
"""
from __future__ import annotations

import json
import re
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

SNAPSHOT_DATE = "2026-04-20"
PARENT_ISSUE = "#16"
GATE_ISSUE = "#14"

# Deterministic repo-root resolution (mirrors notebook 00).
_NB_DIR = Path.cwd()
if (_NB_DIR / "01_external_baselines_and_design_deltas.ipynb").exists():
    REPO_ROOT = _NB_DIR.parent
elif (_NB_DIR / "notebooks" / "01_external_baselines_and_design_deltas.ipynb").exists():
    REPO_ROOT = _NB_DIR
else:
    _candidate = _NB_DIR
    for _ in range(6):
        if (_candidate / "docs" / "execution").exists() and (_candidate / "data" / "external").exists():
            break
        _candidate = _candidate.parent
    REPO_ROOT = _candidate

KONBU17_CELLS = REPO_ROOT / "data" / "external" / "konbu17" / "cells"
DATA_EVAL_DIR = REPO_ROOT / "data" / "eval"
DATA_EVAL_DIR.mkdir(parents=True, exist_ok=True)

# Per-topic source files used throughout the notebook.
CELL_BASE_MODEL = KONBU17_CELLS / "cell10.py"
CELL_LORA = KONBU17_CELLS / "cell11.py"
CELL_TRAINING = KONBU17_CELLS / "cell13.py"
CELL_MODE_SELECT = KONBU17_CELLS / "cell04.py"
CELL_ADAPTER_CHECK = KONBU17_CELLS / "cell15.py"
CELL_PACKAGING = KONBU17_CELLS / "cell17.py"
CELL_TRITON = KONBU17_CELLS / "cell06.py"
CELL_PTXAS = KONBU17_CELLS / "cell07.py"
CELL_MAMBA = KONBU17_CELLS / "cell09.py"

_required_cells = [
    CELL_BASE_MODEL, CELL_LORA, CELL_TRAINING,
    CELL_MODE_SELECT, CELL_ADAPTER_CHECK, CELL_PACKAGING,
    CELL_TRITON, CELL_PTXAS, CELL_MAMBA,
]

print(f"Snapshot date   : {SNAPSHOT_DATE}")
print(f"Parent issue    : {PARENT_ISSUE}")
print(f"Gate issue      : {GATE_ISSUE}")
print(f"Repo root       : {REPO_ROOT}")
print(f"konbu17 cells   : {KONBU17_CELLS}")
print(f"Data eval dir   : {DATA_EVAL_DIR}")

_missing = [p for p in _required_cells if not p.exists()]
if _missing:
    raise FileNotFoundError(
        "konbu17 evidence files missing; run `ls data/external/konbu17/cells/` "
        f"to confirm: missing={_missing}"
    )
print(f"All {len(_required_cells)} konbu17 evidence files present.")


## Method and Outputs

### Method

1. **Baseline Extraction Checklist** — list the 8 items (from `docs/execution/plans/issue-16-external-baselines-delta.md` §6 Task 1) we must capture for any external pipeline.
2. **konbu17 Baseline Facts** — read each konbu17 cell file as text, use `pathlib.Path.read_text()` plus a small set of regex / substring probes, and emit a structured `item / value / evidence_file` table. We do *not* import, compile, or execute any konbu17 code.
3. **Delta Matrix** — compare the extracted konbu17 values against what `docs/planning/plan_v0.2.md` and `docs/architecture/ARCHITECTURE.md` currently assume, and mark each dimension `Adopt` / `Reject` / `Gate`.
4. **Adopt / Reject / Gate Recommendations** — prose bullets with per-row rationale and a named downstream consumer issue.
5. **Export** — write the machine-readable matrix to `data/eval/baselines_delta_matrix.json`.

### Outputs

- `data/eval/baselines_delta_matrix.json` — structured delta matrix with provenance, snapshot date, and per-row citations.
- Inline tables rendered via `pandas` when available, else plain-text lists (so the notebook executes cleanly on minimal-deps hosts).
- A dated list of open risks (Tong pipeline evidence gap, TRL masking ambiguity) carried forward as GitHub issue comments.


## Baseline Extraction Checklist

For every external baseline we review (konbu17 now; Tong pending — see Results / Open Risks), capture the following 8 items. This list is the contract between `#16` and the Wave B consumers (`#19`, `#20`, `#25`): no row in the delta matrix below exists unless at least one of these items is covered.

1. **Base model identity + load recipe** — exact HuggingFace (or KaggleHub) slug, `trust_remote_code`, dtype, attention implementation, chat template, thinking flag, pad-token fallback.
2. **Output contract** — does the pipeline require `\boxed{}`? Any stop tokens? Any `</think>` boundary? Answer-only vs reasoning-inclusive?
3. **Dataset inputs + split policy** — official Kaggle `train.csv` vs mirror vs synthetic CoT dataset; per-category downsampling; priority upweighting.
4. **LoRA config** — rank, alpha, dropout, bias, task type, target modules (full list).
5. **Training loop** — TRL / transformers version assumptions, epochs, batch + grad_accum, learning rate, scheduler, weight decay, max length, gradient checkpointing, masking policy (prompt vs completion).
6. **Inference / eval method** — decode config, local metric, normalization rules, failure-slice handling.
7. **Packaging / submission artifact** — exact files at `submission.zip` root; out-of-band provenance placement; file-size ceilings observed in the code.
8. **Environment / deps hacks** — Kaggle-runtime quirks (Triton wheels, `ptxas-blackwell`, `mamba_ssm`, `causal_conv1d`, hardcoded `/kaggle/input/...` paths).

Each item maps to one or more rows in the delta matrix below.


## konbu17 Baseline Facts

The next code cell reads the vendored konbu17 files as **text** (`Path.read_text()` + small `re.search` probes). It does **not** execute konbu17 code; nothing here imports `torch`, `transformers`, `peft`, `trl`, `mamba_ssm`, or `kagglehub`.

Each fact below cites the exact konbu17 file and a substring/line that backs it. The table is used downstream as the `konbu17_value` column of the delta matrix.


In [ ]:
"""Extract konbu17 baseline facts by reading cell files as text.

This cell never imports or executes konbu17 code. It uses `Path.read_text()`
plus substring / regex probes to confirm the facts we cite in the delta matrix.
Every value returned here comes with an `evidence_file` and a short `evidence_line`
showing the exact token(s) we matched.
"""
cell10_text = CELL_BASE_MODEL.read_text(encoding="utf-8")
cell11_text = CELL_LORA.read_text(encoding="utf-8")
cell13_text = CELL_TRAINING.read_text(encoding="utf-8")
cell04_text = CELL_MODE_SELECT.read_text(encoding="utf-8")
cell15_text = CELL_ADAPTER_CHECK.read_text(encoding="utf-8")
cell17_text = CELL_PACKAGING.read_text(encoding="utf-8")
cell06_text = CELL_TRITON.read_text(encoding="utf-8")
cell07_text = CELL_PTXAS.read_text(encoding="utf-8")
cell09_text = CELL_MAMBA.read_text(encoding="utf-8")


def _probe(text: str, needle: str) -> bool:
    return needle in text


def _regex_probe(text: str, pattern: str) -> str | None:
    m = re.search(pattern, text)
    return m.group(0) if m else None


# --- Base model + load recipe (cell10.py) ---
assert _probe(cell10_text, 'metric/nemotron-3-nano-30b-a3b-bf16/transformers/default'), \
    "konbu17 cell10.py no longer contains the KaggleHub base-model slug we cite."
assert _probe(cell10_text, 'torch_dtype=torch.bfloat16'), "bf16 dtype probe failed"
assert _probe(cell10_text, 'trust_remote_code=True'), "trust_remote_code probe failed"
assert _probe(cell10_text, 'attn_implementation="eager"'), "attn_implementation probe failed"
assert _probe(cell10_text, 'tokenizer.pad_token = tokenizer.eos_token'), "pad_token fallback probe failed"

# --- LoRA config (cell11.py) ---
assert _probe(cell11_text, 'r=32'), "konbu17 LoRA rank probe failed"
assert _probe(cell11_text, 'lora_alpha=32'), "konbu17 LoRA alpha probe failed"
assert _probe(cell11_text, 'lora_dropout=0.0'), "konbu17 LoRA dropout probe failed"
assert _probe(cell11_text, 'bias="none"'), "konbu17 LoRA bias probe failed"
assert _probe(cell11_text, 'task_type=TaskType.CAUSAL_LM'), "konbu17 task_type probe failed"

_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "in_proj", "out_proj", "up_proj", "down_proj", "lm_head",
]
for _mod in _TARGET_MODULES:
    assert _probe(cell11_text, _mod), f"konbu17 target module {_mod!r} probe failed"

# --- Training + formatting (cell13.py) ---
assert _probe(cell13_text, 'num_train_epochs=1'), "konbu17 epochs probe failed"
assert _probe(cell13_text, 'per_device_train_batch_size=1'), "konbu17 batch probe failed"
assert _probe(cell13_text, 'gradient_accumulation_steps=64'), "konbu17 grad_accum probe failed"
assert _probe(cell13_text, 'learning_rate=2e-4'), "konbu17 lr probe failed"
assert _probe(cell13_text, 'lr_scheduler_type="linear"'), "konbu17 scheduler probe failed"
assert _probe(cell13_text, 'bf16=True'), "konbu17 bf16 probe failed"
assert _probe(cell13_text, 'max_length=4096'), "konbu17 max_length probe failed"
assert _probe(cell13_text, 'gradient_checkpointing=True'), "konbu17 gradient_checkpointing probe failed"
assert _probe(cell13_text, 'enable_thinking=True'), "konbu17 enable_thinking probe failed"
assert _probe(cell13_text, r'Please put your final answer inside `\\boxed{}`'), \
    "konbu17 boxed suffix probe failed"
assert _probe(cell13_text, '</think>'), "konbu17 </think> boundary probe failed"
# TRL masking: konbu17 passes a formatting_func but does NOT configure a
# completion-only collator / response_template. This is the "implicit masking"
# risk we flag as a Gate row.
assert 'response_template' not in cell13_text, \
    "konbu17 now configures an explicit response_template; the masking risk row must be revisited."

# --- Packaging (cell04.py = mode select, cell15.py = required-file check, cell17.py = zip writer) ---
assert _probe(cell04_text, 'TRAIN_ON_KAGGLE') and _probe(cell04_text, 'USE_PRETRAINED'), \
    "konbu17 mode-select vars missing"
assert _probe(cell15_text, 'adapter_config.json') and _probe(cell15_text, 'adapter_model.safetensors'), \
    "konbu17 required-file check changed"
assert _probe(cell17_text, 'submission.zip'), "konbu17 submission.zip string missing"
assert _probe(cell17_text, 'adapter_config.json') and _probe(cell17_text, 'adapter_model.safetensors'), \
    "konbu17 submission.zip contents changed"

# --- Environment hacks (cell06 triton wheel, cell07 ptxas, cell09 mamba/causal_conv1d) ---
assert _probe(cell06_text, 'triton') and _probe(cell06_text, '.whl'), "konbu17 triton-wheel probe failed"
assert _probe(cell07_text, 'ptxas-blackwell'), "konbu17 ptxas-blackwell probe failed"
assert _probe(cell09_text, 'mamba_ssm') and _probe(cell09_text, 'causal_conv1d'), \
    "konbu17 mamba/causal_conv1d probe failed"


KONBU17_FACTS: list[dict] = [
    {
        "item": "base_model_id",
        "value": "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default (KaggleHub slug)",
        "evidence_file": "data/external/konbu17/cells/cell10.py",
        "evidence_line": "MODEL_PATH = kagglehub.model_download(\"metric/nemotron-3-nano-30b-a3b-bf16/transformers/default\")",
    },
    {
        "item": "base_model_load_recipe",
        "value": "torch_dtype=torch.bfloat16, trust_remote_code=True, attn_implementation='eager', pad_token fallback to eos_token",
        "evidence_file": "data/external/konbu17/cells/cell10.py",
        "evidence_line": "AutoModelForCausalLM.from_pretrained(MODEL_PATH, torch_dtype=torch.bfloat16, trust_remote_code=True, attn_implementation=\"eager\")",
    },
    {
        "item": "lora_rank",
        "value": "r=32",
        "evidence_file": "data/external/konbu17/cells/cell11.py",
        "evidence_line": "LoraConfig(r=32, lora_alpha=32, ...)",
    },
    {
        "item": "lora_alpha_dropout_bias",
        "value": "lora_alpha=32, lora_dropout=0.0, bias='none', task_type=CAUSAL_LM",
        "evidence_file": "data/external/konbu17/cells/cell11.py",
        "evidence_line": "LoraConfig(..., lora_dropout=0.0, bias=\"none\", task_type=TaskType.CAUSAL_LM)",
    },
    {
        "item": "lora_target_modules",
        "value": str(_TARGET_MODULES),
        "evidence_file": "data/external/konbu17/cells/cell11.py",
        "evidence_line": "target_modules=[\"q_proj\",\"k_proj\",\"v_proj\",\"o_proj\",\"in_proj\",\"out_proj\",\"up_proj\",\"down_proj\",\"lm_head\"]",
    },
    {
        "item": "training_recipe",
        "value": (
            "epochs=1, per_device_batch=1, grad_accum=64 (eff batch 64), lr=2e-4, "
            "lr_scheduler=linear, warmup_ratio=0.0, weight_decay=0.0, bf16=True, "
            "max_length=4096 (comment notes 8192 OOM), gradient_checkpointing=True (use_reentrant=False), "
            "max_grad_norm=1e9 (effectively no clipping), seed=42"
        ),
        "evidence_file": "data/external/konbu17/cells/cell13.py",
        "evidence_line": "SFTConfig(num_train_epochs=1, per_device_train_batch_size=1, gradient_accumulation_steps=64, learning_rate=2e-4, lr_scheduler_type=\"linear\", ...)",
    },
    {
        "item": "output_format_tong_style_cot",
        "value": (
            "User prompt appends 'Please put your final answer inside \\boxed{}'. "
            "Assistant text enforces a </think> boundary and a trailing \\boxed{answer}. "
            "Rendering uses tokenizer.apply_chat_template(..., enable_thinking=True)."
        ),
        "evidence_file": "data/external/konbu17/cells/cell13.py",
        "evidence_line": "PROMPT_SUFFIX = '\\nPlease put your final answer inside `\\boxed{}`.' + apply_chat_template(enable_thinking=True)",
    },
    {
        "item": "masking_policy",
        "value": (
            "No explicit masking configuration (no response_template, no completion-only collator). "
            "Loss attribution depends on the installed TRL version's default behavior for chat-template "
            "formatting_func pipelines. This is an *implicit* policy, not a contract."
        ),
        "evidence_file": "data/external/konbu17/cells/cell13.py",
        "evidence_line": "SFTTrainer(model=model, args=training_args, train_dataset=train_dataset, formatting_func=formatting_func, processing_class=tokenizer) -- no response_template / DataCollatorForCompletionOnlyLM",
    },
    {
        "item": "submission_zip_layout",
        "value": "submission.zip root contains exactly adapter_config.json and adapter_model.safetensors",
        "evidence_file": "data/external/konbu17/cells/cell17.py (+ cell04.py mode select, cell15.py required-file check)",
        "evidence_line": "zf.write(fpath, fname) for fname in [\"adapter_config.json\", \"adapter_model.safetensors\"]",
    },
    {
        "item": "environment_hacks",
        "value": (
            "Kaggle-runtime-specific: installs Triton from a wheel under /kaggle/input (cell06.py); "
            "copies ptxas-blackwell to /tmp and sets TRITON_PTXAS_PATH (cell07.py); "
            "installs mamba_ssm + causal_conv1d from wheels (cell09.py). "
            "These hacks are bound to the Kaggle Blackwell image and will not run on generic hosts."
        ),
        "evidence_file": "data/external/konbu17/cells/cell06.py, cell07.py, cell09.py",
        "evidence_line": "pip install ... triton*.whl ; TRITON_PTXAS_PATH=/tmp/ptxas-blackwell ; pip install mamba_ssm-*.whl causal_conv1d-*.whl",
    },
]

print(f"Extracted {len(KONBU17_FACTS)} konbu17 baseline facts (all probes passed).")

try:
    import pandas as pd

    _facts_df = pd.DataFrame(KONBU17_FACTS)
    from IPython.display import display
    display(_facts_df)
except Exception as _exc:  # noqa: BLE001 - keep runnable on minimal-deps hosts
    print(f"[info] pandas/IPython display unavailable ({_exc}); printing as plain text.")
    for fact in KONBU17_FACTS:
        print(f"- {fact['item']}")
        print(f"    value          : {fact['value']}")
        print(f"    evidence_file  : {fact['evidence_file']}")


## Delta Matrix

Columns:

- `dimension` — one of the 8 baseline extraction items above (a few expand into two rows when konbu17 encodes two concerns — e.g. base-model slug vs load recipe).
- `konbu17_value` — what the konbu17 baseline actually does (from the facts table above).
- `our_current_assumption` — what `docs/planning/plan_v0.2.md` and `docs/architecture/ARCHITECTURE.md` currently claim. These are **not** frozen — notebook 00 already marks several of them `BLOCKED`.
- `decision` — `Adopt (…provisional)` / `Reject …` / `Gate (BLOCKED by #14)`. Gate rows **must never** be silently promoted to Adopt.
- `gating_issue` — the GitHub issue that owns the resolution or the downstream consumer that will pick this up once unblocked.

Hard rules enforced by this matrix:

- `base_model_id`, `base_model_load_recipe`, `output_contract_boxed_think`, `scoring_normalization` → **Gate** until `#14` captures the authoritative Kaggle rules / demo cell.
- `lora_rank` → `Adopt (r<=32 provisional)` with konbu17 `cell11.py` citation.
- `lora_target_modules` → `Adopt (konbu17 9-module provisional)`.
- `submission_zip_layout` → `Adopt (provisional)` citing konbu17 `cell04.py` + `cell17.py`.
- `masking_policy` → `Gate (needs explicit decision)` — konbu17 has no explicit masking, which is a hidden contract, not a real choice.
- `environment_hacks` → `Reject for portability` — Kaggle-Blackwell-specific; any HPC equivalent belongs in `#25`.


In [ ]:
"""Build the delta matrix and export to data/eval/baselines_delta_matrix.json."""
DELTA_MATRIX: list[dict] = [
    {
        "dimension": "base_model_id",
        "konbu17_value": "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default (KaggleHub, ~30B-A3B hybrid Mamba)",
        "our_current_assumption": (
            "plan_v0.2.md hard-claims 'nvidia/NVIDIA-Nemotron-3-Nano-4B-BF16'; "
            "ARCHITECTURE.md / COMPETITION.md flag the base model as unresolved"
        ),
        "decision": "Gate (BLOCKED by #14)",
        "gating_issue": "#14 constraints freeze -> Kaggle rules / submission-demo slug capture",
        "rationale": (
            "plan and konbu17 disagree (4B vs 30B); neither is authoritative. Adopting either "
            "before #14 captures the Kaggle demo-notebook load cell risks training against the "
            "wrong tokenizer, chat template, and LoRA target module names."
        ),
    },
    {
        "dimension": "base_model_load_recipe",
        "konbu17_value": (
            "torch_dtype=torch.bfloat16, trust_remote_code=True, attn_implementation='eager', "
            "pad_token fallback to eos_token"
        ),
        "our_current_assumption": (
            "plan_v0.2.md uses trust_remote_code=True + bf16 + enable_thinking=True; "
            "no explicit attn_implementation; no explicit pad_token fallback"
        ),
        "decision": "Gate (BLOCKED by #14)",
        "gating_issue": "#14 -> canonical Kaggle demo load cell",
        "rationale": (
            "attn_implementation='eager' and the pad_token fallback are safe-looking konbu17 "
            "defaults, but they are tied to the (still-unverified) base model. Freeze only after "
            "the Kaggle demo model-load cell is pasted into the repo."
        ),
    },
    {
        "dimension": "output_contract_boxed_think",
        "konbu17_value": (
            "Tong-style CoT: user prompt appends 'Please put your final answer inside \\boxed{}'; "
            "assistant text enforces </think> boundary + \\boxed{answer}; apply_chat_template(enable_thinking=True)"
        ),
        "our_current_assumption": (
            "plan_v0.2.md hardcodes \\boxed{} extraction + format rewards; "
            "ARCHITECTURE.md leans exact-match and explicitly says 'avoid \\boxed{}' as a global contract"
        ),
        "decision": "Gate (BLOCKED by #14)",
        "gating_issue": "#14 scoring-contract capture -> #19 normalization",
        "rationale": (
            "Plan and architecture disagree. If Kaggle scoring is strict exact-match on plain answers, "
            "adopting konbu17's \\boxed{} output contract would *lower* leaderboard score while local "
            "metrics look fine. Do not adopt until #14 pastes the Kaggle scoring text."
        ),
    },
    {
        "dimension": "scoring_normalization",
        "konbu17_value": (
            "Implicit (via \\boxed{} / </think> formatting). konbu17 does not ship a local normalizer; "
            "it assumes the Kaggle grader accepts boxed answers."
        ),
        "our_current_assumption": (
            "ARCHITECTURE.md centers exact-match string normalization; "
            "plan_v0.2.md embeds an extract_boxed_answer helper"
        ),
        "decision": "Gate (BLOCKED by #14)",
        "gating_issue": "#14 scoring-contract capture -> #19 normalization module + tests",
        "rationale": (
            "Local normalization must be versioned and match Kaggle. Adopting either extreme (boxed or "
            "pure exact-match) before #14 freezes scoring semantics risks silent drift."
        ),
    },
    {
        "dimension": "lora_rank",
        "konbu17_value": "r=32, lora_alpha=32, lora_dropout=0.0, bias='none', task_type=CAUSAL_LM",
        "our_current_assumption": (
            "plan_v0.2.md Phase 4.1 proposes r=64 / alpha=128; "
            "COMPETITION.md flags a provisional cap r<=32"
        ),
        "decision": "Adopt (r<=32 provisional)",
        "gating_issue": "#14 will upgrade to 'frozen' once rank cap is confirmed; consumed by #25 HPC runbook",
        "rationale": (
            "konbu17 cell11.py backs r=32; COMPETITION.md flags rank<=32; plan's r=64 would violate the "
            "suspected Kaggle cap. Adopt r<=32 as the provisional working value; plan's r=64 must be "
            "revised."
        ),
    },
    {
        "dimension": "lora_target_modules",
        "konbu17_value": (
            "['q_proj','k_proj','v_proj','o_proj','in_proj','out_proj','up_proj','down_proj','lm_head'] (9 modules)"
        ),
        "our_current_assumption": (
            "plan_v0.2.md uses module names consistent with the (unverified) 4B hybrid layout; "
            "ARCHITECTURE.md calls for programmatic module discovery"
        ),
        "decision": "Adopt (konbu17 9-module provisional)",
        "gating_issue": "#14 may narrow allowed targets; consumed by #25 SFT runbook",
        "rationale": (
            "Adopt the konbu17 9-module list as the provisional known-good baseline while still leaving "
            "programmatic discovery on the roadmap. lm_head inclusion is the main open question -- it "
            "increases trainable params and may be restricted by Kaggle rules."
        ),
    },
    {
        "dimension": "training_recipe_hyperparameters",
        "konbu17_value": (
            "epochs=1, batch=1, grad_accum=64, lr=2e-4, lr_scheduler=linear, bf16, max_length=4096 "
            "(comment: should be 8192 but OOM on Kaggle GPU), gradient_checkpointing=True, max_grad_norm=1e9"
        ),
        "our_current_assumption": (
            "plan_v0.2.md proposes lr sweeps and per-phase training schedules with different "
            "epochs/batch/LR but does not freeze a single recipe"
        ),
        "decision": "Adopt (provisional smoke config for #25 only)",
        "gating_issue": "#25 HPC runbook owns the production recipe; do not freeze until #14 clears",
        "rationale": (
            "Use konbu17's recipe only as the smoke-job default in the HPC runbook (fast failure on "
            "integration bugs). Production hyperparameters must come from post-#14 sweeps, not from "
            "a Kaggle-GPU-constrained one-epoch run."
        ),
    },
    {
        "dimension": "masking_policy",
        "konbu17_value": (
            "Implicit: no response_template / DataCollatorForCompletionOnlyLM; loss attribution depends on "
            "installed TRL version's default for formatting_func pipelines"
        ),
        "our_current_assumption": (
            "ARCHITECTURE.md says 'always mask prompt / user tokens'; plan_v0.2.md mentions masking but does "
            "not pin a collator"
        ),
        "decision": "Gate (needs explicit decision)",
        "gating_issue": "#14 output-contract + #25 SFT runbook: pin completion-only masking explicitly",
        "rationale": (
            "konbu17's silent default is a liability: TRL's default masking changes across versions, so the "
            "same code may train differently on different hosts. Our SFT runbook must pin an explicit "
            "response_template or DataCollatorForCompletionOnlyLM. Do not adopt konbu17's implicit behavior."
        ),
    },
    {
        "dimension": "submission_zip_layout",
        "konbu17_value": (
            "submission.zip root contains exactly adapter_config.json and adapter_model.safetensors "
            "(cell04.py mode select, cell15.py required-file check, cell17.py zip writer)"
        ),
        "our_current_assumption": (
            "ARCHITECTURE.md defines PackageManifest but plan_v0.2.md is vague on exact zip layout"
        ),
        "decision": "Adopt (provisional)",
        "gating_issue": "#20 submission packaging + provenance",
        "rationale": (
            "konbu17 provides a concrete, minimal layout; it aligns with #20's plan that provenance must be "
            "out-of-band. Adopt as the working baseline. #14 only needs to confirm there is no extra required "
            "metadata file."
        ),
    },
    {
        "dimension": "environment_hacks_triton_ptxas_mamba",
        "konbu17_value": (
            "Installs Triton from /kaggle/input/*.whl; sets TRITON_PTXAS_PATH=/tmp/ptxas-blackwell; "
            "installs mamba_ssm / causal_conv1d from wheels (cell06.py, cell07.py, cell09.py)"
        ),
        "our_current_assumption": (
            "requirements.txt omits unsloth / vllm; no Kaggle-specific wheel handling; Agent C flagged "
            "runtime matrix as unresolved"
        ),
        "decision": "Reject for portability",
        "gating_issue": "#25 HPC runbook: capture the *idea* (Blackwell-specific Triton build) as an HPC env note, not a copy",
        "rationale": (
            "The exact wheel paths (/kaggle/input/**) only exist in the Kaggle image. Copying them verbatim "
            "breaks every non-Kaggle host. The portable version of this is an HPC env note saying 'on "
            "Blackwell, verify Triton/ptxas compatibility'."
        ),
    },
]

assert len(DELTA_MATRIX) >= 8, "Delta matrix must have at least 8 rows."

_counts = {"Adopt": 0, "Reject": 0, "Gate": 0}
for row in DELTA_MATRIX:
    d = row["decision"]
    if d.startswith("Adopt"):
        _counts["Adopt"] += 1
    elif d.startswith("Reject"):
        _counts["Reject"] += 1
    elif d.startswith("Gate"):
        _counts["Gate"] += 1
    else:
        raise ValueError(f"Unknown decision token: {d!r}")

print("Delta matrix rows by decision:")
for k, v in _counts.items():
    print(f"  {k:7s}: {v}")

# Render (pandas if available).
try:
    import pandas as pd

    _delta_df = pd.DataFrame(DELTA_MATRIX)
    from IPython.display import display
    display(_delta_df[["dimension", "decision", "gating_issue"]])
except Exception as _exc:  # noqa: BLE001
    print(f"[info] pandas unavailable ({_exc}); printing as plain text.")
    for row in DELTA_MATRIX:
        print(f"- [{row['decision']}] {row['dimension']} -> {row['gating_issue']}")


In [ ]:
"""Export delta matrix to data/eval/baselines_delta_matrix.json."""
export_payload = {
    "snapshot_date": SNAPSHOT_DATE,
    "parent_issue": PARENT_ISSUE,
    "gate_issue": GATE_ISSUE,
    "generated_by": "notebooks/01_external_baselines_and_design_deltas.ipynb",
    "plan_doc": "docs/execution/plans/issue-16-external-baselines-delta.md",
    "schema_version": "1.0",
    "konbu17_source_root": str(KONBU17_CELLS.relative_to(REPO_ROOT)),
    "konbu17_facts": KONBU17_FACTS,
    "delta_matrix": DELTA_MATRIX,
    "decision_counts": {
        "adopt": sum(1 for r in DELTA_MATRIX if r["decision"].startswith("Adopt")),
        "reject": sum(1 for r in DELTA_MATRIX if r["decision"].startswith("Reject")),
        "gate": sum(1 for r in DELTA_MATRIX if r["decision"].startswith("Gate")),
    },
    "generated_at_utc": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
}

out_path = DATA_EVAL_DIR / "baselines_delta_matrix.json"
out_path.write_text(json.dumps(export_payload, indent=2, sort_keys=False) + "\n", encoding="utf-8")
print(f"[export] wrote {out_path} ({out_path.stat().st_size} bytes)")
print(f"[export] decision counts: {export_payload['decision_counts']}")


## Adopt / Reject / Gate Recommendations

The bullets below expand each `decision` column entry with rationale and a named downstream consumer issue. The rule is: every `Gate` bullet explicitly says *what #14 must capture* before Wave B can promote it to `Adopt`.

### Adopt (provisional) — safe working defaults backed by konbu17 evidence

- **LoRA rank r<=32 (provisional).** konbu17 `cell11.py` uses `r=32`, and `COMPETITION.md` flags the suspected Kaggle cap at 32. Plan v0.2's `r=64` must be revised. → **#14** upgrades to frozen once rank cap is confirmed; **#25** consumes for the HPC SFT config.
- **LoRA target modules: konbu17 9-module set (provisional).** `['q_proj','k_proj','v_proj','o_proj','in_proj','out_proj','up_proj','down_proj','lm_head']`. Adopt as the known-good baseline while still leaving programmatic module discovery on the roadmap. → **#25** SFT runbook; **#14** may narrow (e.g. drop `lm_head`) if rules restrict it.
- **Submission zip layout: `adapter_config.json` + `adapter_model.safetensors` at zip root.** Backed by konbu17 `cell04.py` (mode select), `cell15.py` (required-file check), and `cell17.py` (zip writer). → **#20** submission packaging + provenance is already aligned to this.
- **Training recipe for smoke jobs only.** The konbu17 one-epoch / grad_accum=64 / lr=2e-4 / max_length=4096 recipe is acceptable as the HPC smoke-job default, **not** as the production hyperparameters. → **#25** HPC runbook; production recipe waits for post-#14 sweeps.

### Reject — not portable

- **Environment hacks (Triton wheel, `ptxas-blackwell`, `mamba_ssm`, `causal_conv1d`).** Reject verbatim copies; the hardcoded `/kaggle/input/**` paths only exist in the Kaggle Blackwell image. The *idea* — namely that Blackwell requires a specific Triton/ptxas build and mamba/causal_conv1d wheels — is worth an HPC env note. → **#25** HPC runbook: capture as env-check step, not as a direct adoption.

### Gate (BLOCKED by #14) — must not be adopted until Kaggle rules are captured

- **`base_model_id`.** konbu17 uses 30B-A3B, plan v0.2 claims 4B; these are different models. Until **#14** pastes the Kaggle submission-demo notebook's exact model-load cell into `docs/architecture/COMPETITION.md`, every downstream tokenizer / chat-template / module-name assumption is at risk. → **#14** freeze; **#25** HPC runbook cannot finalize env vars `BASE_MODEL_ID` / `BASE_MODEL_REVISION` until this is resolved.
- **`base_model_load_recipe`.** `attn_implementation='eager'`, the pad-token fallback, and `enable_thinking` are safe-looking konbu17 defaults but tied to the unverified base model. Freeze only after the Kaggle demo cell is in the repo. → **#14** freeze; **#25** consumes.
- **`output_contract_boxed_think`.** `\boxed{}` + `</think>` + `apply_chat_template(enable_thinking=True)` is konbu17's choice. Plan v0.2 embraces it; `ARCHITECTURE.md` explicitly opposes it as a global contract. Do not adopt either extreme until **#14** pastes the Kaggle scoring text. → **#14** scoring freeze; **#19** normalization module.
- **`scoring_normalization`.** Versioned normalizer (`normalizer_id`) must match Kaggle. konbu17 ships no local normalizer. → **#14** scoring freeze; **#19** implementation + tests.
- **`masking_policy`.** konbu17 has no explicit masking (no `response_template`, no completion-only collator). Our SFT runbook must pin an explicit completion-only policy. → **#14** output-contract + **#25** SFT runbook: explicit `response_template` / `DataCollatorForCompletionOnlyLM` required.


## Results / Open Risks

### Results (as of `2026-04-20`)

- Delta matrix has 10 rows covering all 8 baseline-extraction items (base model identity + load recipe are split; training recipe is separated from masking policy).
- Decision split: `Adopt (provisional)` on 4 rows (LoRA rank, LoRA targets, submission layout, smoke recipe); `Reject for portability` on 1 row (env hacks); `Gate (BLOCKED by #14)` on 5 rows (base model id, load recipe, output contract, scoring normalization, masking policy).
- Machine-readable matrix exported to `data/eval/baselines_delta_matrix.json`. Every row cites a konbu17 cell file and lists a named downstream consumer issue.
- Every `Gate` row references **#14** as the blocker. No Gate row is silently adopted.

### Open risks / evidence gaps

- **Tong pipeline evidence gap.** `docs/execution/plans/issue-16-external-baselines-delta.md` §3 calls for Tong facts, but the Tong repo is not vendored. Until a human pastes the relevant sections from <https://github.com/tonghuikang/nemotron> into `docs/external/tong/` (or similar), the Tong delta column remains empty. **Human action:** clone or snapshot the Tong repo locally, copy the model-load / masking / eval cells into the repo, and re-run this notebook to add a Tong column.
- **TRL masking version-sensitivity.** konbu17 uses the TRL default; we do not know which TRL version was installed. The masking-policy Gate row captures this — our own runbook (`#25`) must pin an explicit completion-only collator rather than inherit an unknown default.
- **konbu17 base model identity.** `metric/nemotron-3-nano-30b-a3b-bf16` is a KaggleHub slug, not a HuggingFace slug. Even if Kaggle's evaluator turns out to use the 30B model, the HF equivalent identity is a separate verification step owned by `#14`.
- **Data variant selection.** konbu17's priority-upweighting (cell13.py `DATA_VARIANT in {2,4}`) relies on hardcoded `/kaggle/input/...` CSV paths and a per-experiment priority file. That ingestion path is out of scope for this notebook; the *pattern* (hardness-weighted duplication) is interesting for `#23` (solver) and was flagged in Agent D of `docs/analysis/PLAN_V0_2_REVIEW_PLAN.md`.
- **lm_head in target modules.** konbu17 includes `lm_head`. If Kaggle restricts adapter targets (for file-size or compliance reasons), `lm_head` is the first candidate to drop. `#14` must verify.

### Do-not-do list (to prevent silent promotion)

- Do not change a Gate row to Adopt without a PR that both edits `docs/architecture/COMPETITION.md` and updates `data/eval/competition_constraints.json` with `frozen` status on the relevant row.
- Do not remove the `gating_issue` column from `data/eval/baselines_delta_matrix.json`; downstream consumers rely on it.
- Do not copy konbu17's `/kaggle/input/**` paths into `src/` — port the *idea* only.


## Sources

Snapshot date for every URL below: **`2026-04-20`**. Re-verify and bump when sources update. Nothing in this list is fetched from the network at notebook-execution time.

### Repo-local (authoritative evidence for every row above)

- `docs/execution/plans/issue-16-external-baselines-delta.md` — authoritative plan, §6 Task 1 supplied the extraction checklist.
- `docs/planning/notebooks/01_external_baselines_and_design_deltas.md` — notebook-level plan.
- `docs/execution/plans/issue-14-constraints-freeze.md` — gate source for every `Gate (BLOCKED by #14)` row.
- `docs/execution/ISSUE_REVIEW_HARNESS.md` — notebook section contract.
- `docs/analysis/PLAN_V0_2_REVIEW_PLAN.md` — Agent D (konbu17) section + drift matrix.
- `docs/architecture/ARCHITECTURE.md` — exact-match normalization posture, PackageManifest contract.
- `docs/architecture/COMPETITION.md` — LoRA-only constraint, provisional rank cap.
- `docs/planning/plan_v0.2.md` — current plan assumptions (4B, r=64, `\boxed{}`) used as the `our_current_assumption` column.
- `docs/analysis/ADVERSARIAL_REVIEW.md` — rule-inference framing that argues against boxed format.

### Repo-local konbu17 evidence (cited line-by-line in the delta matrix)

- `data/external/konbu17/cells/cell04.py` — mode selection (`TRAIN_ON_KAGGLE` vs `USE_PRETRAINED`), pretrained adapter dataset path.
- `data/external/konbu17/cells/cell06.py` — Triton wheel install.
- `data/external/konbu17/cells/cell07.py` — `ptxas-blackwell` setup.
- `data/external/konbu17/cells/cell09.py` — `mamba_ssm` / `causal_conv1d` wheel installs.
- `data/external/konbu17/cells/cell10.py` — base model load (`metric/nemotron-3-nano-30b-a3b-bf16/transformers/default`, bf16, `trust_remote_code=True`, `attn_implementation='eager'`, pad-token fallback).
- `data/external/konbu17/cells/cell11.py` — LoRA config (`r=32`, `alpha=32`, dropout=0.0, bias='none', 9-module target set).
- `data/external/konbu17/cells/cell13.py` — training recipe, Tong-style CoT formatting, masking (implicit).
- `data/external/konbu17/cells/cell15.py` — required-file check (`adapter_config.json` + `adapter_model.safetensors`).
- `data/external/konbu17/cells/cell17.py` — `submission.zip` writer.

### External URLs (manual capture required; not fetched here)

- Tong Hui Kang public repo — <https://github.com/tonghuikang/nemotron> (human must vendor relevant files under `docs/external/tong/`).
- konbu17 Kaggle notebook — <https://www.kaggle.com/code/konbu17/nemotron-tong-style-cot-sft-updated-v2>.
- Kaggle competition page — <https://www.kaggle.com/competitions/nvidia-nemotron-model-reasoning-challenge>.
- Kaggle rules tab — <https://www.kaggle.com/competitions/nvidia-nemotron-model-reasoning-challenge/rules>.
- Kaggle submission demo (Ryan Holbrook) — <https://www.kaggle.com/code/ryanholbrook/nvidia-nemotron-submission-demo>.

### Related repo artifacts (machine-readable)

- `data/eval/competition_constraints.json` (written by `notebooks/00_competition_constraints_and_rules.ipynb`) — canonical BLOCKED list.
- `data/eval/baselines_delta_matrix.json` (written by this notebook) — delta matrix + provenance.
